# Cross-Harness Middleware Demo

This notebook shows that `AnonymizationMiddleware` and `SensitivityRouterMiddleware` — built for LangChain agents — run **unmodified** in DeerFlow profiles. No harness-neutral middleware wrapper exists or is needed: DeerFlow's embedded client already forwards LangChain `AgentMiddleware` instances to the underlying `DeerFlowClient`.

See: [docs/middleware-pii-and-routing.md](../docs/middleware-pii-and-routing.md), [docs/harness.md](../docs/harness.md).

## 1. Anonymization middleware — standalone (no agent)

`AnonymizationMiddleware` wraps a thread-scoped `AnonymizationConfig` + Presidio detector. Below we exercise the same low-level `anonymize_text()` helper the middleware itself calls, to see the detect → replace → mapping flow explicitly.

In [ ]:
from faker import Faker

from genai_tk.extra.nlp import PresidioDetector, PresidioDetectorConfig
from genai_tk.extra.nlp.anonymization import anonymize_text

detector = PresidioDetector(config=PresidioDetectorConfig(analyzed_fields=["PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER"]))
faker = Faker()
faker.seed_instance(42)

text = "Hi, I'm Alice Martin, reach me at alice.martin@example.com."
anonymized, mapping = anonymize_text(text, detector=detector, faker=faker)
print("Original :", text)
print("Anonymized:", anonymized)
print("Mapping   :", mapping)

## 2. Building the middleware instances

These are the exact classes registered in `genai_tk.agents.langchain.middleware` — no DeerFlow-specific subclass exists.

In [ ]:
from genai_tk.agents.langchain.middleware import AnonymizationMiddleware, SensitivityRouterMiddleware

anonymization_mw = AnonymizationMiddleware(
    analyzed_fields=["PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER", "CREDIT_CARD"],
    faker_seed=42,
    fuzzy_deanonymize=True,
)

router_mw = SensitivityRouterMiddleware(
    safe_llm="parrot_local@fake",  # swap for a real, cheaper/safer model in production
    sensitive_source_patterns=["**/hr/**", "**/confidential/**"],
)

print(type(anonymization_mw).__name__, type(router_mw).__name__)

## 3. Same middleware, LangChain agent

`MiddlewareConfig` uses the shape `{"class": "<qualified.dotted.Name>", **kwargs}`. Passing the middlewares list directly to `AgentProfileConfig` — or via YAML — instantiates identically.

In [ ]:
from genai_tk.agents.harness.langchain_harness import LangChainHarness
from genai_tk.agents.langchain.config import AgentProfileConfig, MiddlewareConfig

langchain_profile = AgentProfileConfig(
    name="privacy-safe-react",
    type="react",
    llm="parrot_local@fake",
    middlewares=[
        MiddlewareConfig(
            **{
                "class": "genai_tk.agents.langchain.middleware.anonymization_middleware.AnonymizationMiddleware",
                "faker_seed": 42,
            }
        ),
    ],
)

lc_harness = LangChainHarness(langchain_profile)
print("LangChain profile middlewares:", [m.class_path for m in langchain_profile.middlewares])

## 4. Same middleware, DeerFlow profile

`DeerFlowProfile.middlewares` uses the **exact same** `MiddlewareConfig` model as LangChain profiles (upgraded from a plain list of qualified names specifically so it could carry kwargs like `analyzed_fields` / `safe_llm`).

In [ ]:
from genai_tk.agents.deer_flow.profile import DeerFlowProfile

deerflow_profile = DeerFlowProfile.model_validate(
    {
        "name": "Privacy-Safe Research",
        "mode": "thinking",
        "middlewares": [
            {
                "class": "genai_tk.agents.langchain.middleware.anonymization_middleware.AnonymizationMiddleware",
                "analyzed_fields": ["PERSON", "EMAIL_ADDRESS"],
                "faker_seed": 42,
            },
            {
                "class": (
                    "genai_tk.agents.langchain.middleware.sensitivity_router_middleware.SensitivityRouterMiddleware"
                ),
                "safe_llm": "parrot_local@fake",
            },
        ],
    }
)
print(f"harness={deerflow_profile.harness!r}")
for mw in deerflow_profile.middlewares:
    print(" -", mw.class_path, mw.extra_kwargs)

In [ ]:
# The DeerFlow CLI/harness builder instantiates them the same way the LangChain
# factory does (instantiate_middlewares), plus prepends RichToolCallMiddleware.
from genai_tk.agents.deer_flow.cli_commands import _build_cli_middlewares

instantiated = _build_cli_middlewares(deerflow_profile.middlewares)
for mw in instantiated:
    print(type(mw).__name__)

## 5. Running the DeerFlow profile through the unified harness

```python
from genai_tk.agents.harness import create_harness

harness = create_harness("Privacy-Safe Research")  # resolves to DeerFlowHarness
async for event in harness.astream("My email is alice@example.com, research AI safety"):
    ...
```

Requires the `harnessing` extra (`uv sync --extra harnessing`) and a `config/agents/deerflow.yaml`
with the `Privacy-Safe Research` profile (see `config/examples/agents/deerflow.yaml` for a working example).

## Caveat: multi-node graphs

DeerFlow's graph may invoke the model multiple times per turn (planner → researcher → reporter, …).
`AnonymizationMiddleware` tracks already-anonymized message ids per `thread_id`
(`_anonymized_msg_ids`) so intermediate model calls within the same run don't
re-anonymize or leak PII. This is exercised in
`tests/unit_tests/agents/deer_flow/test_middleware_integration.py`.